# Narrador de Cenas — Kinetics + CNN Residual + ONNX
### Adaptado do notebook da aula (PyTorch → ONNX) para demo com webcam na sala

Pipeline da aula: **Dados → Limpeza → Augmentation → CNN Residual → Treino → Teste → Inferência → ONNX**

Dataset estratégico (ações de sala):
- Kinetics-700 (`opening door`, `closing door`, …) — **porta não existe no Kinetics-400**
- `walking the dog` → classe `walking` (narração: pessoa andando)
- Clipes próprios: `standing_up` / `sitting_down` (levantar / sentar)

> Objetivo: gerar `narrador_cenas.onnx` e usar no Streamlit (arquivo ou **webcam**).


## 1. Instalação e configuração


In [ ]:
!pip install -q imagehash onnx onnxscript onnxruntime opencv-python-headless scikit-learn matplotlib pillow tqdm yt-dlp

In [ ]:
import os, random, warnings, json
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

## 2. Dados (Kinetics subset + custom)

No Cursor/terminal (recomendado fora do Colab):

```bash
# Opção A — YouTube (pode pedir cookies / falhar em cloud)
python scripts/download_kinetics_subset.py --per-class-train 10 --per-class-val 2 --per-class-test 2

# Opção B — dataset demo sintético (valida o pipeline; troque antes da aula real)
python scripts/generate_demo_dataset.py

# Clipes sentar/levantar: veja scripts/prepare_custom_actions.md
python scripts/build_frames_dataset.py
```

Abaixo assumimos `data/frames/{train,val,test}/<classe>/*.jpg`.


In [ ]:
PROJECT = Path('.').resolve()
FRAMES_ROOT = PROJECT / 'data' / 'frames'
MODELS_DIR = PROJECT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

assert (FRAMES_ROOT / 'train').exists(), 'Rode scripts/build_frames_dataset.py antes'

for split in ['train', 'val', 'test']:
    n = len(list((FRAMES_ROOT / split).rglob('*.jpg')))
    print(f'{split}: {n} frames')
print('Classes:', sorted(p.name for p in (FRAMES_ROOT/'train').iterdir() if p.is_dir()))

## 3. Limpeza rápida e amostras


In [ ]:
def verificar_imagens(diretorio):
    rem = []
    for root, _, files in os.walk(diretorio):
        for fname in files:
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            caminho = os.path.join(root, fname)
            try:
                with Image.open(caminho) as img:
                    img.verify()
                with Image.open(caminho) as img:
                    img.load()
            except Exception:
                rem.append(caminho); os.remove(caminho)
    return rem

for nome in ['train', 'val', 'test']:
    print(nome, len(verificar_imagens(FRAMES_ROOT/nome)), 'removidas')

classes = sorted(p.name for p in (FRAMES_ROOT/'train').iterdir() if p.is_dir())
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, cls in zip(axes.ravel(), classes):
    imgs = list((FRAMES_ROOT/'train'/cls).glob('*.jpg'))
    ax.imshow(Image.open(random.choice(imgs)))
    ax.set_title(cls, fontsize=9); ax.axis('off')
plt.suptitle('Amostras — Kinetics/custom (frames)', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Augmentation, DataLoaders e CNN Residual (aula)


In [ ]:
IMG_SIZE, BATCH_SIZE = 128, 32
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((int(IMG_SIZE*1.15), int(IMG_SIZE*1.15))),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

ds_train = ImageFolder(FRAMES_ROOT/'train', transform=transform_train)
ds_val = ImageFolder(FRAMES_ROOT/'val', transform=transform_eval)
ds_test = ImageFolder(FRAMES_ROOT/'test', transform=transform_eval)
loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False)
loader_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False)
classes = ds_train.classes
print('Classes:', classes)
print(f'Treino={len(ds_train)} Val={len(ds_val)} Teste={len(ds_test)}')

In [ ]:
class BlocoResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.bloco = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(out_ch),
        )
        self.projetor = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))
                         if stride != 1 or in_ch != out_ch else nn.Identity())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bloco(x) + self.projetor(x))

class CNNResidual(nn.Module):
    def __init__(self, n_classes=8, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))
        self.features = nn.Sequential(
            BlocoResidual(64, 64), BlocoResidual(64, 128, 2), BlocoResidual(128, 256, 2))
        self.classificador = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(256, n_classes))
    def forward(self, x):
        return self.classificador(self.features(self.stem(x)))

modelo = CNNResidual(n_classes=len(classes)).to(device)
print('Parâmetros:', sum(p.numel() for p in modelo.parameters() if p.requires_grad))

## 5. Treinamento (AdamW, label smoothing, early stopping)


In [ ]:
LR, NUM_EPOCHS, PACIENCIA = 3e-4, 15, 5
otimizador = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
criterio = nn.CrossEntropyLoss(label_smoothing=0.1)

def treinar_epoca(modelo, loader):
    modelo.train(); loss_t=ok=n=0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        otimizador.zero_grad(); logits=modelo(x); loss=criterio(logits,y)
        loss.backward(); torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0); otimizador.step()
        loss_t += loss.item()*x.size(0); ok += (logits.argmax(1)==y).sum().item(); n += x.size(0)
    return loss_t/n, 100*ok/n

@torch.no_grad()
def avaliar(modelo, loader):
    modelo.eval(); loss_t=ok=n=0; P=[]; Y=[]
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        logits=modelo(x); loss=criterio(logits,y); pred=logits.argmax(1)
        loss_t += loss.item()*x.size(0); ok += (pred==y).sum().item(); n += x.size(0)
        P.append(pred.cpu()); Y.append(y.cpu())
    return loss_t/n, 100*ok/n, torch.cat(P).numpy(), torch.cat(Y).numpy()

historico = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
melhor=float('inf'); stale=0; CKPT=MODELS_DIR/'melhor_modelo.pth'
for epoca in range(1, NUM_EPOCHS+1):
    tr_l,tr_a = treinar_epoca(modelo, loader_train)
    va_l,va_a,_,_ = avaliar(modelo, loader_val)
    for k,v in zip(historico, [tr_l,tr_a,va_l,va_a]): historico[k].append(v)
    mark=''
    if va_l < melhor:
        melhor, stale = va_l, 0; torch.save(modelo.state_dict(), CKPT); mark='*'
    else:
        stale += 1
    print(f'Ep {epoca:02d} | T {tr_l:.4f}/{tr_a:.1f}% | V {va_l:.4f}/{va_a:.1f}% {mark}')
    if stale >= PACIENCIA:
        print('Early stopping'); break
modelo.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
print('Checkpoint restaurado.')

In [ ]:
ep = range(1, len(historico['train_loss'])+1)
fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].plot(ep, historico['train_loss'], label='Treino'); axes[0].plot(ep, historico['val_loss'], label='Val'); axes[0].legend(); axes[0].set_title('Loss')
axes[1].plot(ep, historico['train_acc'], label='Treino'); axes[1].plot(ep, historico['val_acc'], label='Val'); axes[1].legend(); axes[1].set_title('Accuracy')
plt.tight_layout(); plt.show()
te_l, te_a, preds, labels = avaliar(modelo, loader_test)
print(f'Teste loss={te_l:.4f} acc={te_a:.2f}%')
print(classification_report(labels, preds, target_names=classes, zero_division=0))
fig, ax = plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay(confusion_matrix(labels, preds), display_labels=classes).plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
plt.tight_layout(); plt.show()

## 6. Narração a partir de vídeo / frame (webcam)

Templates em português alinhados às 8 classes. No app Streamlit use o modo **Webcam**.


In [ ]:
NARRATION = {
    'opening_door': 'Na cena, alguém está abrindo a porta.',
    'closing_door': 'Na cena, alguém está fechando a porta.',
    'walking': 'Na cena, uma pessoa está andando.',
    'clapping': 'Na cena, alguém está batendo palmas.',
    'stretching_arm': 'Na cena, alguém está alongando o braço.',
    'pushing_cart': 'Na cena, alguém está empurrando um carrinho.',
    'standing_up': 'Na cena, uma pessoa está se levantando da cadeira.',
    'sitting_down': 'Na cena, uma pessoa está se sentando.',
}

def narrar_video(caminho, modelo, classes, device, every_n=10, max_frames=20):
    cap = cv2.VideoCapture(str(caminho)); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    preds=[]; idx=0; modelo.eval()
    while len(preds) < max_frames:
        ok, frame = cap.read()
        if not ok: break
        if idx % every_n == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            tensor = transform_eval(Image.fromarray(rgb)).unsqueeze(0).to(device)
            with torch.no_grad():
                probs = F.softmax(modelo(tensor), 1).squeeze().cpu()
            i = int(probs.argmax())
            preds.append({'t': idx/fps, 'class': classes[i], 'conf': float(probs[i]),
                          'text': NARRATION.get(classes[i], classes[i])})
        idx += 1
    cap.release()
    from collections import Counter
    cnt = Counter(p['class'] for p in preds if p['conf'] >= 0.3)
    if not cnt: return 'Cena indefinida.', preds
    dom = cnt.most_common(1)[0][0]
    avg = np.mean([p['conf'] for p in preds if p['class']==dom])
    return f"{NARRATION.get(dom, dom)} (confiança média: {avg:.0%})", preds

demo = next((PROJECT/'sample_data').glob('*.mp4'), None) or next((PROJECT/'data'/'kinetics_subset'/'test').rglob('*.mp4'))
resumo, det = narrar_video(demo, modelo, classes, device)
print('Vídeo:', demo.name); print('Narração:', resumo)
for d in det[:6]:
    print(f"  {d['t']:.1f}s → {d['class']} ({d['conf']:.0%})")

## 7. Exportação ONNX (+ metadados)


In [ ]:
import onnx
modelo.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
onnx_path = MODELS_DIR / 'narrador_cenas.onnx'
torch.onnx.export(modelo, dummy, str(onnx_path), input_names=['imagem'], output_names=['predicoes'],
                  opset_version=18, dynamo=False)
m = onnx.load(str(onnx_path))
meta = {
    'task': 'action_recognition_frame',
    'classes': json.dumps(classes),
    'img_size': str(IMG_SIZE),
    'mean': json.dumps(MEAN),
    'std': json.dumps(STD),
    'color_mode': 'RGB',
    'framework': 'PyTorch',
    'architecture': 'CNNResidual',
    'dataset': 'kinetics_subset_classroom',
}
for k,v in meta.items():
    p = m.metadata_props.add(); p.key, p.value = k, v
onnx.save(m, str(onnx_path))
print(f'ONNX: {onnx_path} ({onnx_path.stat().st_size/1e6:.2f} MB)')

## 8. App Streamlit + Webcam + Telegram

```bash
streamlit run app/streamlit_app.py
```

1. Modo **Arquivo**: escolha `opening_door.mp4` / `standing_up.mp4` em `sample_data/`
2. Modo **Webcam**: tire uma foto apontando para a porta / cadeira / pessoa andando
3. **Monitoramento**: token + chat id do Telegram

### Demo sugerida na aula
1. Porta abrindo → “abrindo a porta”
2. Pessoa andando → “pessoa andando”
3. Levantar da cadeira → “levantando da cadeira”
4. Bater palmas → “batendo palmas”

**Nota:** `opening door` / `closing door` estão no **Kinetics-700** (não no 400). Sentar/levantar = clips próprios (`prepare_custom_actions.md`).
